# ARGUS Phase 3: Fine-tune Classifier & UEBA Evaluation

Uses the pre-trained MLM checkpoint and labeled attack sessions to train a binary classifier, then runs the alert engine to evaluate composite severity.

This notebook is a Phase 3 fine-tune smoke/eval run, not a final production validation.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

DATA_ROOT = Path("/kaggle/input/datasets/nightingale21/argus-tokenized-58day-verified/data")
VOCAB_PATH = DATA_ROOT / "tokenized" / "vocab.json"
VAL_MANIFEST = DATA_ROOT / "tokenized" / "sessions_val.pt"

REDTEAM_PATH = Path("/kaggle/input/datasets/nightingale21/attacker/redteam.txt")

REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = Path("/kaggle/working/argus-log-intelligence-platform")
REFRESH_REPO = True

EVAL_CHECKPOINT_DIR = Path("/kaggle/working/argus_mlm_eval_check")
CHECKPOINT_DIR = Path("/kaggle/working/argus_mlm_checkpoints")

ATTACK_MANIFEST = Path("/kaggle/working/attack_sessions/attack_sessions.pt")
NORMAL_SCORES_CSV = Path("/kaggle/working/argus_val_scores_sample.csv")

FINETUNE_OUT = Path("/kaggle/working/argus_finetuned")
PHASE3_REPORT = Path("/kaggle/working/argus_phase3_report.json")

In [ ]:
def run_stream(command, cwd=None, env=None):
    print("$", " ".join(str(p) for p in command), flush=True)
    proc = subprocess.Popen(
        [str(p) for p in command],
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc})")

import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

if REFRESH_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run_stream(["git", "clone", REPO_URL, str(REPO_DIR)])

run_stream([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers>=4.37.0",
    "pyarrow>=14.0.0",
    "tqdm>=4.67.1",
    "pyyaml>=6.0",
    "scikit-learn",
])
print("Repo ready:", REPO_DIR)

## Step 1: Locate Checkpoint & Attack Data

In [ ]:
eval_ckpts = sorted(EVAL_CHECKPOINT_DIR.glob("checkpoint_step_*.pt")) if EVAL_CHECKPOINT_DIR.exists() else []
train_ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint_step_*.pt")) if CHECKPOINT_DIR.exists() else []
checkpoint_candidates = eval_ckpts or train_ckpts
if not checkpoint_candidates:
    raise FileNotFoundError(
        f"No MLM checkpoint found under {EVAL_CHECKPOINT_DIR} or {CHECKPOINT_DIR}. "
        "Upload/unzip the Phase 2 checkpoint archive first."
    )

BEST_CKPT = checkpoint_candidates[-1]
print("MLM checkpoint:", BEST_CKPT)

required_paths = {
    "validation manifest": VAL_MANIFEST,
    "attack manifest": ATTACK_MANIFEST,
}
for label, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    print(f"{label}: {path}")

if NORMAL_SCORES_CSV.exists():
    print("normal scores:", NORMAL_SCORES_CSV)
else:
    print("normal scores not found; continuing because fine-tuning only needs manifests")

## Step 2: Fine-tune Classifier

In [ ]:
finetune_cmd = [
    sys.executable,
    "-m",
    "src.training.finetune",
    "--normal-manifest", str(VAL_MANIFEST),
    "--attack-manifest", str(ATTACK_MANIFEST),
    "--checkpoint", str(BEST_CKPT),
    "--out", str(FINETUNE_OUT),
    "--epochs", "20",
    "--lr", "1e-5",
    "--batch-size", "64",
    "--max-normal", "10000",
    "--freeze-layers", "4",
    "--val-split", "0.2",
    "--limit-chunks", "20",
    "--num-workers", "2",
]
run_stream(
    finetune_cmd,
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": str(REPO_DIR)},
)
print("Fine-tuning complete")

## Step 3: Evaluate with Alert Engine

In [ ]:
sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd

from src.inference.alert_engine import AlertEngine, ScoredSession

best_ckpt = FINETUNE_OUT / "best_classifier.pt"
if not best_ckpt.exists():
    raise FileNotFoundError(f"No fine-tuned classifier checkpoint found: {best_ckpt}")

checkpoint = torch.load(best_ckpt, map_location="cpu", weights_only=False)
print(f"Best classifier: epoch {checkpoint['epoch']}, F1={checkpoint['best_f1']:.4f}")
print(f"Val metrics: {checkpoint['val_metrics']}")

history_path = FINETUNE_OUT / "finetune_history.json"
if history_path.exists():
    history = json.loads(history_path.read_text())
    print(f"\nTraining history ({len(history)} epochs):")
    print(f"{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10} {'Val F1':>8} {'Prec':>8} {'Recall':>8}")
    for h in history:
        print(
            f"{h['epoch']:>5} {h['train_loss']:>12.4f} {h['loss']:>10.4f} "
            f"{h['f1']:>8.4f} {h['precision']:>8.4f} {h['recall']:>8.4f}"
        )
else:
    print("No fine-tuning history found:", history_path)

In [ ]:
engine = AlertEngine()

attack_sessions = [
    ScoredSession(
        session_id=f"attack_{i}",
        user_id=f"user_{i % 5:02x}",
        host_id=f"host_{i % 3:02x}",
        anomaly_score=7.0 + np.random.rand(),
        classification="attack",
        classification_confidence=0.8 + np.random.rand() * 0.15,
        technique_id="T1078",
    )
    for i in range(20)
]

normal_sessions = [
    ScoredSession(
        session_id=f"normal_{i}",
        user_id=f"nuser_{i % 10:02x}",
        host_id=f"nhost_{i % 5:02x}",
        anomaly_score=1.0 + np.random.rand() * 2,
        classification="normal",
        classification_confidence=0.95,
        technique_id=None,
    )
    for i in range(50)
]

all_sessions = attack_sessions + normal_sessions
alerts = engine.process_batch(all_sessions)
alert_stats = engine.get_stats()

print(f"Sessions processed: {len(all_sessions)}")
print(f"Alerts generated: {len(alerts)}")
print()

for alert in alerts[:5]:
    print(
        f"  [{alert.alert_class:>8}] {alert.alert_id} user={alert.user_id} "
        f"anomaly={alert.anomaly_score:.2f} composite={alert.composite_severity:.3f}"
    )

print(f"\nEngine stats: {alert_stats}")

## Step 4: Save Phase 3 Report & Archive

In [ ]:
report = {
    "phase": "3",
    "status": "fine_tune_smoke_eval_complete",
    "mlm_checkpoint": str(BEST_CKPT),
    "classifier_checkpoint": str(best_ckpt),
    "best_f1": checkpoint.get("best_f1", 0),
    "val_metrics": checkpoint.get("val_metrics", {}),
    "alert_engine_stats": alert_stats,
}
PHASE3_REPORT.write_text(json.dumps(report, indent=2))
print(f"Report: {PHASE3_REPORT}")

ARCHIVE = Path("/kaggle/working/argus_phase3_archive")
ARCHIVE.mkdir(exist_ok=True)
for file_path in [PHASE3_REPORT, best_ckpt, history_path]:
    if file_path.exists():
        shutil.copy2(file_path, ARCHIVE / file_path.name)

archive = shutil.make_archive(str(ARCHIVE), "zip", root_dir=ARCHIVE)
print(f"Archive: {archive}")
print("Phase 3 fine-tune smoke/eval complete")